In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, StandardScaler

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv("shared_data/train_clean.csv")
val = pd.read_csv("shared_data/val_clean.csv")
test = pd.read_csv("shared_data/test_clean.csv")
target = "is_canceled"

X_train, y_train = train.drop(columns=[target]), train[target]
X_val, y_val = val.drop(columns=[target]), val[target]
X_test, y_test = test.drop(columns=[target]), test[target]

assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)
assert X_train.shape[1] == 24, "RQ1 A requires the shared 24-feature dataset"
for split_name, features, labels in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test),
]:
    assert not features.isna().any().any(), f"Missing values in {split_name}"
    assert set(labels.unique()) == {0, 1}, f"Expected binary labels in {split_name}"
    print(f"{split_name}: {features.shape}")

hidden_sizes = [16, 32, 64]
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
history_cols = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
]

nn = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42,
    early_stopping=False,
    tol=1e-4,
    n_iter_no_change=10,
    verbose=False,
)

In [ ]:
def make_nn_preprocessor(features):
    """Infer feature groups from training columns only, as in the original NN."""
    numeric_cols = features.select_dtypes(include="number").columns.tolist()
    categorical_cols = [col for col in features.columns if col not in numeric_cols]
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ])


def positive_probability(model, features):
    positive_index = np.flatnonzero(model.classes_ == 1).item()
    return model.predict_proba(features)[:, positive_index]


def classification_metrics(labels, prediction):
    return {
        "Accuracy": accuracy_score(labels, prediction),
        "Precision": precision_score(labels, prediction, zero_division=0),
        "Recall": recall_score(labels, prediction, zero_division=0),
        "F1": f1_score(labels, prediction, zero_division=0),
    }


def evaluate_probability(labels, probability, threshold):
    """Use the frozen threshold for class metrics; probabilities for ROC-AUC."""
    prediction = (probability >= threshold).astype(int)
    return {
        **classification_metrics(labels, prediction),
        "ROC_AUC": roc_auc_score(labels, probability),
    }


def tune_nn(features_train, labels_train, features_val, labels_val, experiment):
    """This selection function never receives test features or test labels."""
    assert features_train.columns.equals(features_val.columns)
    preprocessor = make_nn_preprocessor(features_train)
    transformed_train = preprocessor.fit_transform(features_train)
    transformed_val = preprocessor.transform(features_val)
    print(f"{experiment}: train {transformed_train.shape}, val {transformed_val.shape}")

    models, rows = {}, []
    for size in hidden_sizes:
        print(f"{experiment}: training {size} hidden units (up to {nn.max_iter} epochs)...", flush=True)
        candidate = clone(nn).set_params(hidden_layer_sizes=(size,))
        with warnings.catch_warnings(record=True) as fit_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            candidate.fit(transformed_train, labels_train)
        convergence_messages = [
            str(warning.message) for warning in fit_warnings
            if issubclass(warning.category, ConvergenceWarning)
        ]
        # Recording must not hide an unresolved warning from the notebook output.
        for warning in fit_warnings:
            print(
                f"WARNING [{experiment}, {size} units] "
                f"{warning.category.__name__}: {warning.message}", flush=True,
            )
        models[size] = candidate
        # Default model.predict cutoff for the hyperparameter search, as in LR/NB.
        train_prediction = candidate.predict(transformed_train)
        val_prediction = candidate.predict(transformed_val)
        val_metrics = classification_metrics(labels_val, val_prediction)
        rows.append({
            "hidden_units": size,
            "max_iter": candidate.max_iter,
            "iterations": candidate.n_iter_,
            "final_training_loss": candidate.loss_,
            "reached_max_iter": candidate.n_iter_ >= candidate.max_iter,
            "stopped_before_cap": candidate.n_iter_ < candidate.max_iter,
            "convergence_warning": bool(convergence_messages),
            "convergence_warning_messages": " | ".join(convergence_messages),
            "train_f1": f1_score(labels_train, train_prediction, zero_division=0),
            **{f"val_{name.lower()}": value for name, value in val_metrics.items()},
        })
        print(f"Completed: {candidate.n_iter_} epochs, validation F1 {val_metrics['F1']:.4f}, "
              f"stopped before cap={candidate.n_iter_ < candidate.max_iter}, "
              f"ConvergenceWarning={bool(convergence_messages)}", flush=True)

    tuning = pd.DataFrame(rows)
    # idxmax returns the first maximum, so the grid order defines tie-breaking.
    best_units = int(tuning.loc[tuning["val_f1"].idxmax(), "hidden_units"])
    model = models[best_units]
    val_probability = positive_probability(model, transformed_val)
    threshold_tuning = pd.DataFrame([
        {"Threshold": threshold,
         **evaluate_probability(labels_val, val_probability, threshold)}
        for threshold in THRESHOLDS
    ])
    threshold = float(threshold_tuning.loc[threshold_tuning["F1"].idxmax(), "Threshold"])
    print(f"Selected for {experiment}: hidden units={best_units}, threshold={threshold}")
    return {
        "preprocessor": preprocessor,
        "model": model,
        "models": models,
        "hidden_units": best_units,
        "threshold": threshold,
        "tuning": tuning,
        "threshold_tuning": threshold_tuning,
        "val_probability": val_probability,
        "n_train": len(labels_train),
        "n_val": len(labels_val),
        "n_features": features_train.shape[1],
    }


def save_tuning(experiment, suffix=""):
    prefix = "neural_network" + (f"_{suffix}" if suffix else "")
    experiment["tuning"].to_csv(output_dir / f"{prefix}_hidden_units_tuning.csv", index=False)
    experiment["threshold_tuning"].to_csv(output_dir / f"{prefix}_threshold_tuning.csv", index=False)

In [ ]:
main_experiment = tune_nn(X_train, y_train, X_val, y_val, "All 24 features")
best_nn = main_experiment["model"]
best_hidden_units = main_experiment["hidden_units"]
final_threshold = main_experiment["threshold"]
nn_preprocessor = main_experiment["preprocessor"]
nn_tuning_results = main_experiment["tuning"]
nn_threshold_results = main_experiment["threshold_tuning"]
save_tuning(main_experiment)
display(nn_tuning_results.round(4))
display(nn_threshold_results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(nn_tuning_results["hidden_units"], nn_tuning_results["train_f1"], marker="o", label="Training F1")
axes[0].plot(nn_tuning_results["hidden_units"], nn_tuning_results["val_f1"], marker="o", label="Validation F1")
axes[0].set(xlabel="Number of hidden units", ylabel="F1 score", title="Neural network: hidden-layer selection")
axes[0].set_xticks(hidden_sizes)
for metric in ["Precision", "Recall", "F1"]:
    axes[1].plot(nn_threshold_results["Threshold"], nn_threshold_results[metric], marker="o", label=metric)
axes[1].set(xlabel="Classification threshold", ylabel="Score", title="Neural network: validation threshold selection")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "neural_network_validation_tuning.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Freeze the selected model and threshold before transforming/evaluating test.
X_test_nn = nn_preprocessor.transform(X_test)
nn_test_probability = positive_probability(best_nn, X_test_nn)
nn_test_prediction = (nn_test_probability >= final_threshold).astype(int)
nn_test_metrics = evaluate_probability(y_test, nn_test_probability, final_threshold)

nn_final_results = pd.DataFrame([{
    "Model": "Neural Network",
    "hidden_units": best_hidden_units,
    "Threshold": final_threshold,
    **nn_test_metrics,
}])
nn_final_results.to_csv(output_dir / "neural_network_final_results.csv", index=False)
display(nn_final_results.round(4))

selected_training_diagnostics = nn_tuning_results.loc[
    nn_tuning_results["hidden_units"] == best_hidden_units
].iloc[0]
nn_configuration = pd.DataFrame([{
    "Model": "Neural Network",
    "Hidden_units": best_hidden_units,
    "Threshold": final_threshold,
    "Selection_metric": "Validation F1; default cutoff then threshold grid",
    "Activation": best_nn.activation,
    "Solver": best_nn.solver,
    "Learning_rate_init": best_nn.learning_rate_init,
    "Max_iter": best_nn.max_iter,
    "Iterations": best_nn.n_iter_,
    "Stopped_before_cap": bool(selected_training_diagnostics["stopped_before_cap"]),
    "Reached_max_iter": bool(selected_training_diagnostics["reached_max_iter"]),
    "Convergence_warning": bool(selected_training_diagnostics["convergence_warning"]),
    "Convergence_warning_messages": selected_training_diagnostics["convergence_warning_messages"],
    "Final_training_loss": best_nn.loss_,
    "Early_stopping": best_nn.early_stopping,
    "Tolerance": best_nn.tol,
    "N_iter_no_change": best_nn.n_iter_no_change,
    "Random_state": best_nn.random_state,
    "Train_rows": len(y_train),
    "Validation_rows": len(y_val),
    "Test_rows": len(y_test),
}])
nn_configuration.to_csv(output_dir / "neural_network_configuration.csv", index=False)

In [ ]:
X_train_no_history = X_train.drop(columns=history_cols)
X_val_no_history = X_val.drop(columns=history_cols)
X_test_no_history = X_test.drop(columns=history_cols)
assert X_train_no_history.shape[1] == 21

no_history_experiment = tune_nn(
    X_train_no_history, y_train, X_val_no_history, y_val,
    "RQ1 B: without history",
)
best_nn_no_history = no_history_experiment["model"]
save_tuning(no_history_experiment, "no_history")
display(no_history_experiment["tuning"].round(4))
display(no_history_experiment["threshold_tuning"].round(4))

X_test_nn_no_history = no_history_experiment["preprocessor"].transform(X_test_no_history)
nn_no_history_probability = positive_probability(best_nn_no_history, X_test_nn_no_history)
nn_rq1_results = pd.DataFrame([
    {
        "Setting": "A: all features",
        "n_features": 24,
        "hidden_units": best_hidden_units,
        "Threshold": final_threshold,
        "n_test": len(y_test),
        **nn_test_metrics,
    },
    {
        "Setting": "B: without history",
        "n_features": 21,
        "hidden_units": no_history_experiment["hidden_units"],
        "Threshold": no_history_experiment["threshold"],
        "n_test": len(y_test),
        **evaluate_probability(y_test, nn_no_history_probability, no_history_experiment["threshold"]),
    },
])
nn_rq1_results.to_csv(output_dir / "neural_network_rq1_results.csv", index=False)
display(nn_rq1_results.round(4))
print("Test A - B ROC-AUC:", round(nn_rq1_results.loc[0, "ROC_AUC"] - nn_rq1_results.loc[1, "ROC_AUC"], 4))
print("Test A - B F1:", round(nn_rq1_results.loc[0, "F1"] - nn_rq1_results.loc[1, "F1"], 4))
# Per-row predictions allow every RQ1 metric to be reproduced without retraining.
nn_rq1_predictions = pd.concat([
    pd.DataFrame({
        "Setting": setting,
        "test_row_id": X_test.index.to_numpy(),
        "actual": y_test.to_numpy(),
        "probability": probability,
        "predicted": (probability >= threshold).astype(int),
        "Threshold": threshold,
    })
    for setting, probability, threshold in [
        ("A", nn_test_probability, final_threshold),
        ("B", nn_no_history_probability, no_history_experiment["threshold"]),
    ]
], ignore_index=True)
assert len(nn_rq1_predictions) == 2 * len(test)
assert not nn_rq1_predictions.duplicated(["Setting", "test_row_id"]).any()
nn_rq1_predictions.to_csv(output_dir / "neural_network_rq1_predictions.csv", index=False)


In [ ]:
hotels = ["City Hotel", "Resort Hotel"]


def split_by_hotel(frame, hotel):
    subset = frame.loc[frame["hotel"] == hotel]
    return subset.drop(columns=[target, "hotel"]), subset[target]


nn_hotel_experiments = {}
nn_rq2_rows = []
nn_rq2_baseline_rows = []
nn_rq2_prediction_frames = []
for source_hotel in hotels:
    source_X_train, source_y_train = split_by_hotel(train, source_hotel)
    source_X_val, source_y_val = split_by_hotel(val, source_hotel)
    assert source_X_train.shape[1] == 23
    source_baseline = DummyClassifier(strategy="most_frequent")
    source_baseline.fit(source_X_train, source_y_train)
    experiment = tune_nn(
        source_X_train, source_y_train, source_X_val, source_y_val,
        f"RQ2 source: {source_hotel}",
    )
    nn_hotel_experiments[source_hotel] = experiment
    suffix = source_hotel.lower().replace(" ", "_")
    save_tuning(experiment, suffix)
    display(experiment["tuning"].round(4))
    display(experiment["threshold_tuning"].round(4))

    for evaluate_hotel in hotels:
        hotel_X_test, hotel_y_test = split_by_hotel(test, evaluate_hotel)
        transformed_test = experiment["preprocessor"].transform(hotel_X_test)
        probability = positive_probability(experiment["model"], transformed_test)
        source_baseline_prediction = source_baseline.predict(hotel_X_test)
        source_baseline_probability = positive_probability(source_baseline, hotel_X_test)
        nn_rq2_prediction_frames.append(pd.DataFrame({
            "train_hotel": source_hotel,
            "test_hotel": evaluate_hotel,
            # Preserve the original test_clean.csv positions across hotel subsets.
            "test_row_id": hotel_X_test.index.to_numpy(),
            "actual": hotel_y_test.to_numpy(),
            "probability": probability,
            "predicted": (probability >= experiment["threshold"]).astype(int),
            "Threshold": experiment["threshold"],
            "baseline_predicted": source_baseline_prediction,
            "baseline_probability": source_baseline_probability,
        }))
        nn_rq2_baseline_rows.append({
            "train_hotel": source_hotel,
            "evaluate_hotel": evaluate_hotel,
            "Model": "Source training-majority baseline",
            "Decision_rule": "Source training majority class; no NN threshold",
            "predicted_class": int(source_baseline_prediction[0]),
            "n_train": len(source_y_train),
            "n_test": len(hotel_y_test),
            "train_cancel_rate": source_y_train.mean(),
            "test_cancel_rate": hotel_y_test.mean(),
            **classification_metrics(hotel_y_test, source_baseline_prediction),
            "ROC_AUC": roc_auc_score(hotel_y_test, source_baseline_probability),
        })
        nn_rq2_rows.append({
            "train_hotel": source_hotel,
            "evaluate_hotel": evaluate_hotel,
            "evaluation_type": "in-hotel" if source_hotel == evaluate_hotel else "transfer",
            "hidden_units": experiment["hidden_units"],
            "n_features": experiment["n_features"],
            "Threshold": experiment["threshold"],
            "n_train": len(source_y_train),
            "n_val": len(source_y_val),
            "n_test": len(hotel_y_test),
            "train_cancel_rate": source_y_train.mean(),
            "val_cancel_rate": source_y_val.mean(),
            "test_cancel_rate": hotel_y_test.mean(),
            **evaluate_probability(hotel_y_test, probability, experiment["threshold"]),
        })

nn_rq2_results = pd.DataFrame(nn_rq2_rows)
assert len(nn_rq2_results) == 4
assert nn_rq2_results["evaluate_hotel"].notna().all()
nn_rq2_results.to_csv(output_dir / "neural_network_rq2_results.csv", index=False)
display(nn_rq2_results.round(4))
nn_rq2_baseline_results = pd.DataFrame(nn_rq2_baseline_rows)
assert len(nn_rq2_baseline_results) == 4
nn_rq2_baseline_results.to_csv(output_dir / "neural_network_rq2_baseline_results.csv", index=False)
metric_names = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
baseline_columns = nn_rq2_baseline_results[
    ["train_hotel", "evaluate_hotel", *metric_names]
].rename(columns={name: f"Baseline_{name}" for name in metric_names})
nn_rq2_vs_baseline = nn_rq2_results.merge(
    baseline_columns, on=["train_hotel", "evaluate_hotel"], validate="one_to_one",
)
for name in metric_names:
    nn_rq2_vs_baseline[f"Delta_{name}"] = (
        nn_rq2_vs_baseline[name] - nn_rq2_vs_baseline[f"Baseline_{name}"]
    )
nn_rq2_vs_baseline.to_csv(output_dir / "neural_network_rq2_vs_baseline.csv", index=False)
print("Source-fitted majority baselines:")
display(nn_rq2_baseline_results.round(4))
print("Model minus baseline (positive Delta means a higher model score):")
display(nn_rq2_vs_baseline.round(4))

# Two frozen source models each evaluate every test booking exactly once.
nn_rq2_predictions = pd.concat(nn_rq2_prediction_frames, ignore_index=True)
assert len(nn_rq2_predictions) == 2 * len(test)
assert not nn_rq2_predictions.duplicated(["train_hotel", "test_row_id"]).any()
nn_rq2_predictions.to_csv(output_dir / "neural_network_rq2_predictions.csv", index=False)


In [ ]:
nn_rq2_auc = nn_rq2_results.pivot(index="train_hotel", columns="evaluate_hotel", values="ROC_AUC")
ax = nn_rq2_auc.loc[hotels, hotels].plot.bar(figsize=(8, 4), ylim=(0, 1), rot=0)
ax.set(xlabel="Training hotel", ylabel="Test ROC-AUC", title="Neural network: within-hotel and transfer performance")
ax.legend(title="Test hotel")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "neural_network_rq2_roc_auc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
nn_baseline = DummyClassifier(strategy="most_frequent")
nn_baseline.fit(X_train, y_train)
baseline_prediction = nn_baseline.predict(X_test)
baseline_probability = positive_probability(nn_baseline, X_test)
nn_baseline_metrics = {
    **classification_metrics(y_test, baseline_prediction),
    "ROC_AUC": roc_auc_score(y_test, baseline_probability),
}
nn_test_comparison = pd.DataFrame([
    {
        "Model": "Majority baseline",
        "Decision_rule": "Training majority class",
        **nn_baseline_metrics,
    },
    {
        "Model": "Neural Network",
        "Decision_rule": f"Validation-selected threshold {final_threshold:.2f}",
        **nn_test_metrics,
    },
])
nn_test_comparison.to_csv(output_dir / "neural_network_vs_baseline.csv", index=False)
display(nn_test_comparison.round(4))
nn_rq1_vs_baseline = nn_rq1_results.copy()
for name, value in nn_baseline_metrics.items():
    nn_rq1_vs_baseline[f"Baseline_{name}"] = value
    nn_rq1_vs_baseline[f"Delta_{name}"] = nn_rq1_vs_baseline[name] - value
nn_rq1_vs_baseline.to_csv(output_dir / "neural_network_rq1_vs_baseline.csv", index=False)
print("RQ1: both feature settings against the same training-majority baseline")
display(nn_rq1_vs_baseline.round(4))

nn_training_diagnostics = pd.concat([
    experiment["tuning"].assign(
        experiment=name,
        selected=lambda frame: frame["hidden_units"].eq(experiment["hidden_units"]),
    )
    for name, experiment in [
        ("All features", main_experiment),
        ("Without history", no_history_experiment),
        ("City Hotel", nn_hotel_experiments["City Hotel"]),
        ("Resort Hotel", nn_hotel_experiments["Resort Hotel"]),
    ]
], ignore_index=True)
assert len(nn_training_diagnostics) == 12
assert nn_training_diagnostics["max_iter"].eq(1000).all()
nn_training_diagnostics.to_csv(output_dir / "neural_network_training_diagnostics.csv", index=False)
display(nn_training_diagnostics.round(4))
if nn_training_diagnostics["convergence_warning"].any():
    print("LIMITATION: at least one candidate emitted a ConvergenceWarning; see the recorded messages.")
if nn_training_diagnostics["reached_max_iter"].any():
    print("LIMITATION: at least one candidate reached the fixed training cap; optimization may be incomplete.")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, experiment) in zip(axes.flat, [
    ("All features", main_experiment),
    ("Without history", no_history_experiment),
    ("City Hotel", nn_hotel_experiments["City Hotel"]),
    ("Resort Hotel", nn_hotel_experiments["Resort Hotel"]),
]):
    model = experiment["model"]
    ax.plot(range(1, len(model.loss_curve_) + 1), model.loss_curve_)
    ax.set_title(f"{name}: {experiment['hidden_units']} hidden units")
    ax.set_xlabel("Training epoch")
    ax.set_ylabel("Training loss")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "neural_network_training_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
nn_confusion = confusion_matrix(y_test, nn_test_prediction, labels=[0, 1])
pd.DataFrame(
    nn_confusion,
    index=["Actual not canceled", "Actual canceled"],
    columns=["Predicted not canceled", "Predicted canceled"],
).to_csv(output_dir / "neural_network_confusion_matrix.csv")
ConfusionMatrixDisplay(
    confusion_matrix=nn_confusion,
    display_labels=["Not canceled", "Canceled"],
).plot(cmap="Blues", values_format="d")
plt.title(f"Neural Network: Test Confusion Matrix\nThreshold = {final_threshold:.2f}")
plt.tight_layout()
plt.savefig(output_dir / "neural_network_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
nn_error_analysis = X_test.copy()
nn_error_analysis.insert(0, "test_row_id", np.arange(len(X_test), dtype=int))
nn_error_analysis["actual"] = y_test.to_numpy()
nn_error_analysis["predicted"] = nn_test_prediction
nn_error_analysis["cancellation_probability"] = nn_test_probability
# Alias retained alongside the original name for compatibility with LR exports.
nn_error_analysis["cancel_probability"] = nn_error_analysis["cancellation_probability"]
nn_error_analysis["error_type"] = np.select(
    [
        (nn_error_analysis["actual"] == 0) & (nn_error_analysis["predicted"] == 0),
        (nn_error_analysis["actual"] == 0) & (nn_error_analysis["predicted"] == 1),
        (nn_error_analysis["actual"] == 1) & (nn_error_analysis["predicted"] == 0),
        (nn_error_analysis["actual"] == 1) & (nn_error_analysis["predicted"] == 1),
    ],
    ["TN", "FP", "FN", "TP"],
    default="Unknown",
)
nn_error_analysis["threshold"] = final_threshold
nn_error_analysis["probability_minus_threshold"] = nn_test_probability - final_threshold
nn_error_analysis["threshold_distance"] = np.abs(nn_test_probability - final_threshold)
nn_error_analysis["has_booking_history"] = (
    (nn_error_analysis["is_repeated_guest"] > 0)
    | (nn_error_analysis["previous_cancellations"] > 0)
    | (nn_error_analysis["previous_bookings_not_canceled"] > 0)
)
nn_error_analysis["is_error"] = nn_error_analysis["actual"] != nn_error_analysis["predicted"]
nn_error_analysis.to_csv(output_dir / "neural_network_error_analysis.csv", index=False)

numeric_error_features = [
    "lead_time", "total_nights", "total_guests",
    "previous_cancellations", "previous_bookings_not_canceled",
]
nn_error_summary = (
    nn_error_analysis.groupby("error_type")[numeric_error_features]
    .mean()
    .round(2)
)
nn_error_summary.to_csv(output_dir / "neural_network_error_summary.csv")
print("Mean feature values by prediction outcome (same format as LR):")
display(nn_error_summary)

example_cols = [
    "hotel", "lead_time", "total_nights", "total_guests", "deposit_type",
    "market_segment", "customer_type", "previous_cancellations",
    "previous_bookings_not_canceled", "cancel_probability", "actual", "predicted",
    "test_row_id", "is_repeated_guest", "cancellation_probability", "threshold",
    "probability_minus_threshold", "threshold_distance", "error_type",
]
false_negatives = nn_error_analysis.loc[
    (nn_error_analysis["actual"] == 1) & (nn_error_analysis["predicted"] == 0)
]
false_positives = nn_error_analysis.loc[
    (nn_error_analysis["actual"] == 0) & (nn_error_analysis["predicted"] == 1)
]
selected_case_rows = []
for kind, label, errors, ascending, filename in [
    ("FN", "False negatives: lowest cancellation probabilities first", false_negatives, True, "neural_network_FN_examples.csv"),
    ("FP", "False positives: highest cancellation probabilities first", false_positives, False, "neural_network_FP_examples.csv"),
]:
    ranked_errors = errors.sort_values(
        ["cancellation_probability", "test_row_id"], ascending=[ascending, True],
        kind="mergesort",
    )
    examples = ranked_errors.head(20)[example_cols]
    examples.to_csv(output_dir / filename, index=False)
    print(label)
    display(examples.head(5))
    if not ranked_errors.empty:
        selected_case_rows.append(ranked_errors.iloc[0].to_dict() | {"error_type": kind})

nn_selected_error_cases = pd.DataFrame(selected_case_rows)
neighbor_rows = []
if not nn_selected_error_cases.empty:
    # Both preprocessing and this neighbor index are fitted on main TRAIN only.
    neighbor_count = min(5, len(X_train))
    training_neighbors = NearestNeighbors(
        n_neighbors=neighbor_count, metric="euclidean", algorithm="brute",
    ).fit(nn_preprocessor.transform(X_train))
    selected_test_row_ids = nn_selected_error_cases["test_row_id"].astype(int).to_numpy()
    neighbor_distances, neighbor_positions = training_neighbors.kneighbors(
        X_test_nn[selected_test_row_ids],
    )
    for case_position, (_, case) in enumerate(nn_selected_error_cases.iterrows()):
        positions = neighbor_positions[case_position]
        distances = neighbor_distances[case_position]
        neighbor_labels = y_train.iloc[positions].to_numpy()
        nn_selected_error_cases.loc[case_position, "neighbor_count"] = neighbor_count
        nn_selected_error_cases.loc[case_position, "training_neighbor_cancel_rate"] = neighbor_labels.mean()
        nn_selected_error_cases.loc[case_position, "nearest_training_distance"] = distances.min()
        nn_selected_error_cases.loc[case_position, "mean_training_neighbor_distance"] = distances.mean()
        nn_selected_error_cases.loc[case_position, "max_training_neighbor_distance"] = distances.max()
        for rank, (train_position, distance) in enumerate(zip(positions, distances), start=1):
            neighbor_rows.append({
                "test_row_id": int(case["test_row_id"]),
                "error_type": case["error_type"],
                "neighbor_rank": rank,
                "train_row_id": int(train_position),
                "distance": float(distance),
                "neighbor_actual": int(y_train.iloc[train_position]),
                **X_train.iloc[train_position].to_dict(),
            })
    nn_selected_error_cases["main_training_cancel_rate"] = y_train.mean()
    nn_selected_error_cases["evidence_type"] = "Descriptive training-neighbor context; not NN decision explanation"
else:
    print("No false positives or false negatives were observed; no cases to inspect.")

nn_selected_error_cases.to_csv(output_dir / "neural_network_selected_error_cases.csv", index=False)
nn_error_case_neighbors = pd.DataFrame(neighbor_rows)
nn_error_case_neighbors.to_csv(output_dir / "neural_network_error_case_neighbors.csv", index=False)
print("Preselected confident errors with descriptive training-neighbor evidence:")
display(nn_selected_error_cases.round(4))
print("Training row IDs refer to zero-based data rows in shared_data/train_clean.csv.")
print("Neighbor labels are local training context, not a causal or faithful explanation of the NN.")

nn_history_error_summary = (
    nn_error_analysis.groupby("has_booking_history")
    .agg(
        bookings=("is_error", "size"),
        errors=("is_error", "sum"),
        error_rate=("is_error", "mean"),
        cancellation_rate=("actual", "mean"),
    )
    .reset_index()
)
nn_history_error_summary.to_csv(output_dir / "neural_network_history_error_summary.csv", index=False)
display(nn_history_error_summary.round(4))